# Corporate KYC Demo — Part 1: Entity Resolution

This notebook is the reproducible companion to [`demo-story.md`](demo-story.md)
and [`answer-key.md`](answer-key.md). The 20 source documents in
[`documents/`](documents/) were run through Aura Document Intelligence to
produce the graph in this AuraDB instance.

**Starting point for anyone trying this demo:** restore
[`data/db.backup`](data/db.backup) into your own Aura instance, copy
`.env.example` to `.env` and fill in your connection details, then run this
notebook from the top.

**Part 1 (this notebook section): Entity Resolution.** Document
Intelligence extracts entities per document. Feed it several KYC packets
that each mention "Meridian Holdings LLC," and — correctly, by design — you
get several separate mentions of that company, because each document is
processed on its own and there's no way to know a company named in file #3
is the same one named in file #9 without being told. That's true of any
pipeline that extracts entities from a set of documents one at a time, not
a gap specific to this tool. Resolving those mentions into one canonical
node per real-world entity is a standard, expected stage of building a
knowledge graph from documents — and it's a graph problem worth solving
with graph analytics rather than by hand-writing one merge rule per label.

**The approach used below:** build a candidate-match graph among
same-label entity mentions (an edge means "these two mentions share a
strong identifying signal"), then run **Weakly Connected Components**
across the whole thing in one pass. Every mention that's transitively
connected — A matches B on EIN, B matches C on name, so A/B/C cluster
together even though A and C share no signal directly — becomes one
resolved entity. WCC is the graph-analytics primitive doing the actual
work of deciding what counts as "the same thing," not a hand-written
per-label grouping rule.

**Part 2 (next section, TBD): Cypher traversal + GDS algorithms** —
UBO tracing, cycle detection, Louvain community detection, and betweenness
centrality, run against the resolved graph produced here.

In [1]:
import sys, graphdatascience
print(sys.executable)
print(graphdatascience.__version__)

/Users/corydonbaylor/Documents/github/docintel_aga/.venv/bin/python
2.1


In [ ]:
import os
from datetime import timedelta
from dotenv import load_dotenv
from neo4j import GraphDatabase
from graphdatascience.session import AuraAPICredentials, GdsSessions, DbmsConnectionInfo, SessionMemory
import pandas as pd

load_dotenv()

URI = os.environ["NEO4J_URI"]
USERNAME = os.environ["NEO4J_USERNAME"]
PASSWORD = os.environ["NEO4J_PASSWORD"]
DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD), notifications_min_severity="OFF")
driver.verify_connectivity()
print("Connected to", URI, "via the plain Neo4j driver")

# This instance is Aura Graph Analytics: GDS runs as a separate, on-demand
# Session rather than as procedures embedded in the database, so it needs
# its own Aura API credentials (account-level -- create a key pair in the
# Aura Console under your account/organization's "API Keys" settings) in
# addition to the plain DB login above.
SESSION_NAME = "docintel-aga-entity-resolution"

api_credentials = AuraAPICredentials(
    client_id=os.environ["CLIENT_ID"],
    client_secret=os.environ["CLIENT_SECRET"],
    project_id=os.environ.get("TENANT_ID") or None,
)
sessions = GdsSessions(api_credentials=api_credentials)

db_connection = DbmsConnectionInfo(uri=URI, username=USERNAME, password=PASSWORD)

gds = sessions.get_or_create(
    session_name=SESSION_NAME,
    memory=SessionMemory.m_2GB,
    db_connection=db_connection,
    ttl=timedelta(minutes=30),
    neo4j_driver_config={"notifications_min_severity": "OFF"},
)
gds.verify_connectivity()
print("GDS session ready:", SESSION_NAME)

## 1. Baseline: what did Document Intelligence actually give us?

Count nodes by label. Every real entity node is also multi-labeled
`__Entity__` (Document Intelligence's own bookkeeping label), so we filter
that out and take whichever label remains as the domain label.

In [3]:
def label_counts(tx):
    result = tx.run("""
        MATCH (n)
        WHERE NOT n:__Chunk__ AND NOT n:__Document__
        WITH [l IN labels(n) WHERE l <> '__Entity__'][0] AS label
        RETURN label, count(*) AS n
        ORDER BY n DESC
    """)
    return pd.DataFrame([dict(r) for r in result])

with driver.session(database=DATABASE) as session:
    baseline_counts = session.execute_read(label_counts)

baseline_counts

,label,n
0,Organization,51
1,Address,19
2,Person,16
3,RegisteredAgent,15
4,LoanRequest,4
5,AdverseMediaMention,3
6,SanctionsRecord,2


`LoanRequest` and `SanctionsRecord` land at the right count already, because
each was only ever described in a single source document. Organizations,
people, registered agents, and addresses are all fragmented across the
documents that mention them — Meridian Holdings LLC alone shows up as
several separate nodes, one per document that names it.

Let's look at the Organization fragmentation directly.

In [4]:
def duplicate_orgs(tx):
    result = tx.run("""
        MATCH (n:Organization)
        WITH toLower(trim(n.legalName)) AS key, collect(n.legalName)[0] AS name, count(*) AS n
        RETURN name, n
        ORDER BY n DESC
    """)
    return pd.DataFrame([dict(r) for r in result])

with driver.session(database=DATABASE) as session:
    org_groups = session.execute_read(duplicate_orgs)

org_groups

,name,n
0,Meridian Holdings LLC,9
1,Cobalt Trading Co,8
2,Northgate Ventures LLC,6
3,Anchor Point Partners LLC,5
4,Vantage Point Estates LLC,4
5,Cascade Robotics Inc,4
6,Ashford Capital Partners LLC,3
7,Foothill Manufacturing Inc,3
8,Bluewater Logistics Inc,2
9,Redwood Analytics LLC,2


Two rows are worth calling out before we resolve anything:

- **"Meridian Holding Group" (count 1)** — this is *not* a fragment of
  "Meridian Holdings LLC." It's a deliberately similar-but-different entity
  from a sanctions screening record (a false-positive name-similarity hit).
  Resolving these two together would corrupt the graph and manufacture a
  sanctions match that doesn't exist.
- **"Ministry of Finance" (count 1)** — a government body named in Dorian
  Voss's PEP screening record, correctly extracted as its own entity, not a
  fragment of anything.

This is exactly why the candidate-match graph below is built on **exact
matches of hard identifying signals** (EIN, normalized legal name, a
literal application or screening ID) rather than fuzzy name similarity —
fuzzy matching is exactly what would connect "Meridian Holdings LLC" to
"Meridian Holding Group."

## 2. Build the candidate-match graph

For each label, connect two mentions with a `SIMILAR_TO` relationship when
they share a strong identifying signal. This is deliberately conservative:
an edge only exists where we have real corroborating evidence two mentions
refer to the same real-world thing.

| Label | Candidate-match signal |
|---|---|
| Organization | same normalized legal name, **or** same EIN |
| Person | same normalized name |
| RegisteredAgent | same normalized name |
| Address | same normalized street |
| LoanRequest | same application ID |
| SanctionsRecord | same screening ID |
| AdverseMediaMention | same year, and at least one side has no headline (a headline-less stub can only refer to the one dated, headlined story it's clearly describing — not to a second, differently-headlined story that happens to share a year) |

This is a full pairwise comparison per label, which is fine at this scale;
a larger corpus would add blocking keys (e.g. only compare within the same
state or industry) before comparing pairs.

In [5]:
CANDIDATE_MATCH_RULES = {
    "Organization": """
        (a.legalName IS NOT NULL AND toLower(trim(a.legalName)) = toLower(trim(b.legalName)))
        OR (a.ein IS NOT NULL AND a.ein = b.ein)
    """,
    "Person": "a.name IS NOT NULL AND toLower(trim(a.name)) = toLower(trim(b.name))",
    "RegisteredAgent": "a.name IS NOT NULL AND toLower(trim(a.name)) = toLower(trim(b.name))",
    "Address": "a.street IS NOT NULL AND toLower(trim(a.street)) = toLower(trim(b.street))",
    "LoanRequest": "a.applicationId IS NOT NULL AND a.applicationId = b.applicationId",
    "SanctionsRecord": "a.screeningId IS NOT NULL AND a.screeningId = b.screeningId",
    "AdverseMediaMention": "left(a.date, 4) = left(b.date, 4) AND (a.headline IS NULL OR b.headline IS NULL)",
}

def build_candidate_edges(tx, label, condition):
    query = f"""
        MATCH (a:{label}), (b:{label})
        WHERE elementId(a) < elementId(b) AND ({condition})
        MERGE (a)-[:SIMILAR_TO]->(b)
        RETURN count(*) AS edgesCreated
    """
    return tx.run(query).single()["edgesCreated"]

with driver.session(database=DATABASE) as session:
    for label, condition in CANDIDATE_MATCH_RULES.items():
        n_edges = session.execute_write(build_candidate_edges, label, condition)
        print(f"{label:20s} {n_edges:3d} candidate-match edges")

Organization         110 candidate-match edges
Person                20 candidate-match edges
RegisteredAgent       35 candidate-match edges
Address               38 candidate-match edges
LoanRequest            0 candidate-match edges
SanctionsRecord        0 candidate-match edges
AdverseMediaMention    1 candidate-match edges


## 3. Run Weakly Connected Components

This runs on the Aura Graph Analytics **Session** we connected to above
(`gds`), not as procedures embedded in the database -- the session pulls
the graph from AuraDB via a Cypher-based remote projection, runs WCC on
its own compute, and writes results back to AuraDB.

Project the `SIMILAR_TO` candidate-match edges as one graph (emitting both
directions so WCC sees them as undirected), run WCC once across all of
them, and write each node's component ID back as `resolvedComponent`. A
node with no candidate matches (a genuine singleton, or one of the two
lookalike entities above) simply ends up alone in its own component --
which is the correct outcome, not a failure to resolve it.

In [6]:
if gds.graph.exists("entityResolution"):
    gds.graph.drop("entityResolution")

G, project_result = gds.graph.project.cypher(
    "entityResolution",
    """
    CALL () {
        MATCH (a)-[:SIMILAR_TO]->(b)
        RETURN a AS source, b AS target
        UNION
        MATCH (a)-[:SIMILAR_TO]->(b)
        RETURN b AS source, a AS target
    }
    RETURN gds.graph.project.remote(source, target, {
      sourceNodeLabels: labels(source),
      targetNodeLabels: labels(target),
      relationshipType: 'SIMILAR_TO'
    })
    """,
)
print(f"Projected {G.node_count()} nodes, {G.relationship_count()} candidate-match edges (both directions)")

wcc_result = gds.wcc.mutate(G, mutate_property="resolvedComponent")
print(f"WCC found {wcc_result.component_count} components (mutated in-session)")

write_result = gds.graph.node_properties.write(G, ["resolvedComponent"])
print(f"Wrote resolvedComponent back to AuraDB: {write_result}")

G.drop()

Graph creation from Triplets:   0%|          | 0.0/100 [00:00<?, ?%/s]

Projected 91 nodes, 408 candidate-match edges (both directions)
WCC found 22 components (mutated in-session)


Write-Back (graph: entityResolution):   0%|          | 0.0/100 [00:00<?, ?%/s]

Wrote resolvedComponent back to AuraDB: graph_name='entityResolution' node_properties=['resolvedComponent'] properties_written=91 write_millis=274 configuration={'graphName': 'entityResolution', 'nodeProperties': ['resolvedComponent'], 'nodeLabels': ['*'], 'sudo': False, 'logProgress': True, 'jobId': '47ffb801-7117-49ee-bffb-e60c4af69e5d'}


GraphInfo(graph_name='entityResolution', database='neo4j', database_location='remote', configuration={'jobId': 'jid-fe78b825-fce0-4ecc-9d10-41d2b391e8f4', 'validateRelationships': False, 'logProgress': True, 'readConcurrency': 4, 'sudo': False}, memory_usage='', size_in_bytes=-1, node_count=91, relationship_count=408, creation_time=datetime.datetime(2026, 9, 28, 21, 35, 18, 299378, tzinfo=TzInfo(0)), modification_time=datetime.datetime(2026, 9, 28, 21, 35, 19, 506076, tzinfo=TzInfo(0)), graph_schema={'nodes': {'Organization': {'resolvedComponent': 'Integer (DefaultValue(-9223372036854775808), TRANSIENT)'}, 'AdverseMediaMention': {'resolvedComponent': 'Integer (DefaultValue(-9223372036854775808), TRANSIENT)'}, 'Address': {'resolvedComponent': 'Integer (DefaultValue(-9223372036854775808), TRANSIENT)'}, '__Entity__': {'resolvedComponent': 'Integer (DefaultValue(-9223372036854775808), TRANSIENT)'}, 'RegisteredAgent': {'resolvedComponent': 'Integer (DefaultValue(-9223372036854775808), TRANS

## 4. Merge each resolved component into one canonical node

Group nodes by (label, `resolvedComponent`) and merge every group of 2+
with `apoc.refactor.mergeNodes`, which reattaches every relationship from
every duplicate onto the surviving node — this is the step that actually
reconnects the ownership chain across documents. `properties: "discard"`
keeps the first node's property values on conflict, which is simple and
predictable for this demo.

In [7]:
def merge_resolved_components(tx):
    query = """
        MATCH (n)
        WHERE n.resolvedComponent IS NOT NULL
        WITH n, [l IN labels(n) WHERE NOT l STARTS WITH '__'][0] AS label, n.resolvedComponent AS component
        WITH label, component, collect(n) AS nodes
        WHERE size(nodes) >= 2
        CALL apoc.refactor.mergeNodes(nodes, {properties: "discard", mergeRels: true}) YIELD node
        REMOVE node.resolvedComponent
        RETURN label, component, size(nodes) AS mergedFrom
    """
    return list(tx.run(query))

with driver.session(database=DATABASE) as session:
    merges = session.execute_write(merge_resolved_components)
    for m in merges:
        print(f"{m['label']:20s} merged {m['mergedFrom']} nodes -> 1  (component {m['component']})")

Organization         merged 9 nodes -> 1  (component 0)
Organization         merged 8 nodes -> 1  (component 9)
Organization         merged 6 nodes -> 1  (component 17)
Organization         merged 5 nodes -> 1  (component 22)
Organization         merged 3 nodes -> 1  (component 28)
Organization         merged 4 nodes -> 1  (component 31)
Organization         merged 4 nodes -> 1  (component 40)
Organization         merged 3 nodes -> 1  (component 35)
Organization         merged 2 nodes -> 1  (component 38)
Organization         merged 2 nodes -> 1  (component 44)
Organization         merged 2 nodes -> 1  (component 46)
Address              merged 9 nodes -> 1  (component 48)
Address              merged 2 nodes -> 1  (component 57)
Address              merged 2 nodes -> 1  (component 59)
Person               merged 5 nodes -> 1  (component 61)
Person               merged 3 nodes -> 1  (component 66)
Person               merged 4 nodes -> 1  (component 69)
Person               merged 2 nod

## 5. Clean up resolution scaffolding

Drop the `SIMILAR_TO` candidate edges (including any self-loops created
when a merge pulled two matched nodes together) and the leftover
`resolvedComponent` property on nodes that were never part of a multi-node
component — none of this belongs in the target KYC graph model.

In [8]:
def cleanup_scaffolding(tx):
    tx.run("MATCH ()-[r:SIMILAR_TO]->() DELETE r")
    tx.run("MATCH (n) WHERE n.resolvedComponent IS NOT NULL REMOVE n.resolvedComponent")

with driver.session(database=DATABASE) as session:
    session.execute_write(cleanup_scaffolding)

print("Scaffolding removed")

Scaffolding removed


## 6. Drop non-entity extraction noise

Separately from resolving duplicates, two nodes in this graph aren't
mentions of a real entity at all — they're artifacts of the extractor
turning a *narrative description* into a node. Dorian Voss's PEP screening
record says his holdings share "a single registered agent and address"
without naming it, and that phrase became its own `RegisteredAgent` and
`Address` node. These correctly found no candidate match above (we have no
evidence which real agent/address they refer to), but they're also not
useful to keep: they're vague duplicates of facts already captured
correctly elsewhere (Meridian's and Vantage Point's real `USES_AGENT` /
`REGISTERED_AT` edges to Sterling and the Harbor View address). This is a
data-quality cleanup step, distinct from resolution — we're discarding a
non-entity, not merging two mentions of a real one.

In [9]:
def cleanup_vague_entities(tx):
    query = """
        MATCH (n)
        WHERE (n:Address AND (n.street IS NULL OR n.fullAddress =~ '(?i)^(a|an|the)\\s.*'))
           OR (n:RegisteredAgent AND n.name =~ '(?i)^(a|an|the)\\s.*')
        WITH n, coalesce(n.name, n.fullAddress) AS label, labels(n) AS labs
        DETACH DELETE n
        RETURN label, labs
    """
    return list(tx.run(query))

with driver.session(database=DATABASE) as session:
    removed = session.execute_write(cleanup_vague_entities)
    for r in removed:
        print(f"Removed vague {r['labs'][0]} node: {r['label']!r}")

Removed vague Address node: 'a single registered agent and address'
Removed vague RegisteredAgent node: 'a single registered agent'


## 7. Verify

Compare node counts before and after. Organizations should drop to 14 (12
real companies + Meridian Holding Group + Ministry of Finance, correctly
still separate). People should drop to 6. Registered agents to 3. Addresses
to 8.

In [10]:
with driver.session(database=DATABASE) as session:
    resolved_counts = session.execute_read(label_counts)

comparison = baseline_counts.merge(
    resolved_counts, on="label", how="outer", suffixes=("_before", "_after")
).fillna(0)
comparison

,label,n_before,n_after
0,Address,19,8
1,AdverseMediaMention,3,2
2,LoanRequest,4,4
3,Organization,51,14
4,Person,16,6
5,RegisteredAgent,15,3
6,SanctionsRecord,2,2


## 8. Smoke test — does the story hold together now?

Two checks that only pass if entities resolved correctly across documents:

1. **UBO trace**: Dorian Voss never appears on Anchor Point Partners LLC's
   own KYC packet, but should now be reachable through two separate
   ownership paths (via Vantage Point Estates, and via
   Meridian → Northgate → Cobalt).
2. **Ownership cycle**: Meridian → Northgate → Cobalt → Meridian should now
   resolve as a single continuous path instead of disconnected fragments.

In [11]:
def ubo_paths(tx):
    result = tx.run("""
        MATCH p = (voss:Person {name: "Dorian Voss"})-[:OWNS*1..6]->(anchor:Organization {legalName: "Anchor Point Partners LLC"})
        RETURN [n IN nodes(p) | coalesce(n.legalName, n.name)] AS path, length(p) AS hops
        ORDER BY hops
    """)
    return [dict(r) for r in result]

def ownership_cycle(tx):
    result = tx.run("""
        MATCH p = (m:Organization {legalName: "Meridian Holdings LLC"})-[:OWNS*2..5]->(m)
        RETURN [n IN nodes(p) | n.legalName] AS cycle
    """)
    return [dict(r) for r in result]

with driver.session(database=DATABASE) as session:
    print("UBO paths from Dorian Voss to Anchor Point Partners LLC:")
    for row in session.execute_read(ubo_paths):
        print(" ", " -> ".join(row["path"]), f"({row['hops']} hops)")

    print()
    print("Ownership cycles found:")
    for row in session.execute_read(ownership_cycle):
        print(" ", " -> ".join(row["cycle"]))

UBO paths from Dorian Voss to Anchor Point Partners LLC:
  Dorian Voss -> Vantage Point Estates LLC -> Anchor Point Partners LLC (2 hops)
  Dorian Voss -> Meridian Holdings LLC -> Northgate Ventures LLC -> Cobalt Trading Co -> Anchor Point Partners LLC (4 hops)

Ownership cycles found:
  Meridian Holdings LLC -> Northgate Ventures LLC -> Cobalt Trading Co -> Meridian Holdings LLC


Both checks should now succeed — the UBO trace should return two paths, and
the cycle should return one. If they don't, check Section 3's component
count and Section 4's merge log to see what didn't cluster as expected.

Graph is now resolved, with GDS (Weakly Connected Components) doing the
work of deciding what counted as a duplicate. **Next up (Part 2, not yet in
this notebook):** Cypher traversal for the UBO/cycle story above as a demo
beat in its own right, then GDS Louvain (find the ring as a community) and
betweenness centrality (find Priya Nandakumar as the bridge into the
legitimate cluster).

## 9. Tear down the GDS session

Aura Graph Analytics Sessions are billed while they exist (the `ttl` above
also auto-expires it after 30 minutes of inactivity as a safety net, but
it's better to close it explicitly once you're done).

In [12]:
sessions.delete(session_name=SESSION_NAME)
print("Session deleted:", SESSION_NAME)

Session deleted: docintel-aga-entity-resolution
